In [1]:
# ============================================================
# IMPORT LIBRARIES
# ============================================================

import json
import joblib
import numpy as np
import pandas as pd

from pathlib import Path

print("✓ Libraries imported successfully.")

✓ Libraries imported successfully.


In [2]:

# ============================================================
# PROJECT PATHS
# ============================================================

CURRENT_DIR = Path.cwd()
PROJECT_ROOT = CURRENT_DIR.parent

RAW_PATH = PROJECT_ROOT / "data" / "raw"
FEATURE_PATH = PROJECT_ROOT / "data" / "feature_engineered"
OUTPUT_PATH = PROJECT_ROOT / "data" / "eda_outputs"
PREDICTION_PATH = PROJECT_ROOT / "data" / "predictions"
MODEL_PATH = PROJECT_ROOT / "models"

OUTPUT_PATH.mkdir(parents=True, exist_ok=True)
PREDICTION_PATH.mkdir(parents=True, exist_ok=True)

print("Project Root:")
print(PROJECT_ROOT)

Project Root:
C:\Users\Shubham\Desktop\enterprise_hr_ai


In [3]:
# ============================================================
# LOAD EMPLOYEE DATA AND BEST MODEL
# ============================================================

employee_file = RAW_PATH / "employee_attrition.csv"
feature_file = FEATURE_PATH / "employee_features.csv"
model_file = MODEL_PATH / "best_attrition_model.joblib"

if not employee_file.exists():
    raise FileNotFoundError(employee_file)

if not feature_file.exists():
    raise FileNotFoundError(feature_file)

if not model_file.exists():
    raise FileNotFoundError(model_file)

employee_df = pd.read_csv(employee_file)
feature_df = pd.read_csv(feature_file)

best_model = joblib.load(model_file)

print("✓ Employee attrition data loaded.")
print(f"✓ Employee records: {len(employee_df)}")

print("\n✓ Best attrition model loaded.")

✓ Employee attrition data loaded.
✓ Employee records: 1470

✓ Best attrition model loaded.


In [4]:
# ============================================================
# RECREATE MODEL FEATURES
# ============================================================

feature_df["IncomePerYearExperience"] = (
    (feature_df["MonthlyIncome"] * 12)
    / feature_df["TotalWorkingYears"].replace(0, np.nan)
).fillna(0)

feature_df["CompanyTenureRatio"] = (
    feature_df["YearsAtCompany"]
    / feature_df["TotalWorkingYears"].replace(0, np.nan)
).fillna(0)

feature_df["PromotionWaitRatio"] = (
    feature_df["YearsSinceLastPromotion"]
    / feature_df["YearsAtCompany"].replace(0, np.nan)
).fillna(0)

feature_df["OverallSatisfactionScore"] = (
    feature_df["EnvironmentSatisfaction"]
    + feature_df["JobSatisfaction"]
    + feature_df["RelationshipSatisfaction"]
    + feature_df["WorkLifeBalance"]
) / 4

feature_df["CareerStabilityScore"] = (
    feature_df["YearsAtCompany"]
    + feature_df["YearsInCurrentRole"]
    + feature_df["YearsWithCurrManager"]
) / 3


feature_columns = [
    "Age",
    "OverTime",
    "JobSatisfaction",
    "MonthlyIncome",
    "YearsAtCompany",
    "WorkLifeBalance",
    "TotalWorkingYears",
    "YearsInCurrentRole",
    "YearsSinceLastPromotion",
    "YearsWithCurrManager",
    "JobLevel",
    "JobInvolvement",
    "EnvironmentSatisfaction",
    "RelationshipSatisfaction",
    "DistanceFromHome",
    "NumCompaniesWorked",
    "PercentSalaryHike",
    "StockOptionLevel",
    "TrainingTimesLastYear",
    "BusinessTravel",
    "Department",
    "EducationField",
    "JobRole",
    "MaritalStatus",
    "Gender",
    "IncomePerYearExperience",
    "CompanyTenureRatio",
    "PromotionWaitRatio",
    "OverallSatisfactionScore",
    "CareerStabilityScore"
]

X_employee = feature_df[feature_columns].copy()

print(f"✓ Model features prepared: {X_employee.shape[1]}")

✓ Model features prepared: 30


In [5]:
# ============================================================
# EMPLOYEE ATTRITION PREDICTIONS
# ============================================================

attrition_probability = (
    best_model.predict_proba(X_employee)[:, 1]
)

employee_prediction = employee_df.copy()

employee_prediction["Attrition_Prob"] = (
    attrition_probability
)

employee_prediction["Risk"] = pd.cut(
    employee_prediction["Attrition_Prob"],
    bins=[-np.inf, 0.40, 0.70, np.inf],
    labels=["LOW", "MEDIUM", "HIGH"]
)

print("✓ Attrition predictions generated.")

display(
    employee_prediction[
        [
            "EmployeeNumber",
            "Department",
            "JobRole",
            "Attrition_Prob",
            "Risk"
        ]
    ].head(10)
)

✓ Attrition predictions generated.


,EmployeeNumber,Department,JobRole,Attrition_Prob,Risk
0,1,Sales,Sales Executive,0.734188,HIGH
1,2,Research & Development,Research Scientist,0.010894,LOW
2,4,Research & Development,Laboratory Technician,0.571790,MEDIUM
3,5,Research & Development,Research Scientist,0.121609,LOW
4,7,Research & Development,Laboratory Technician,0.375579,LOW
5,8,Research & Development,Laboratory Technician,0.090320,LOW
6,10,Research & Development,Laboratory Technician,0.093720,LOW
7,11,Research & Development,Laboratory Technician,0.145441,LOW
8,12,Research & Development,Manufacturing Director,0.061565,LOW
9,13,Research & Development,Healthcare Representative,0.038739,LOW


In [6]:
# ============================================================
# LOAD DEPARTMENT ENGAGEMENT INTELLIGENCE
# ============================================================

engagement_file = (
    OUTPUT_PATH / "department_engagement_intelligence.csv"
)

if not engagement_file.exists():
    raise FileNotFoundError(engagement_file)

department_engagement = pd.read_csv(
    engagement_file
)

print("✓ Department engagement intelligence loaded.")

display(
    department_engagement[
        [
            "Department",
            "Engagement_Index",
            "Engagement_Level"
        ]
    ]
)

✓ Department engagement intelligence loaded.


,Department,Engagement_Index,Engagement_Level
0,Finance,48.652996,MEDIUM
1,HR,59.836585,MEDIUM
2,IT,48.835476,MEDIUM
3,Marketing,38.153967,LOW
4,Sales,33.478613,LOW


In [7]:
# ============================================================
# ATTACH DEPARTMENT-LEVEL ENGAGEMENT
# ============================================================

employee_intelligence = employee_prediction.merge(
    department_engagement[
        [
            "Department",
            "Engagement_Index",
            "Engagement_Level"
        ]
    ],
    on="Department",
    how="left"
)

print("✓ Department engagement attached to employees.")

display(
    employee_intelligence[
        [
            "EmployeeNumber",
            "Department",
            "JobRole",
            "Attrition_Prob",
            "Risk",
            "Engagement_Index",
            "Engagement_Level"
        ]
    ].head(10)
)

✓ Department engagement attached to employees.


,EmployeeNumber,Department,JobRole,Attrition_Prob,Risk,Engagement_Index,Engagement_Level
0,1,Sales,Sales Executive,0.734188,HIGH,33.478613,LOW
1,2,Research & Development,Research Scientist,0.010894,LOW,NaN,NaN
2,4,Research & Development,Laboratory Technician,0.571790,MEDIUM,NaN,NaN
3,5,Research & Development,Research Scientist,0.121609,LOW,NaN,NaN
4,7,Research & Development,Laboratory Technician,0.375579,LOW,NaN,NaN
5,8,Research & Development,Laboratory Technician,0.090320,LOW,NaN,NaN
6,10,Research & Development,Laboratory Technician,0.093720,LOW,NaN,NaN
7,11,Research & Development,Laboratory Technician,0.145441,LOW,NaN,NaN
8,12,Research & Development,Manufacturing Director,0.061565,LOW,NaN,NaN
9,13,Research & Development,Healthcare Representative,0.038739,LOW,NaN,NaN


In [8]:
# ============================================================
# LOAD ROLE-SKILL INTELLIGENCE
# ============================================================

role_skill_file = (
    OUTPUT_PATH / "role_skill_intelligence.csv"
)

role_summary_file = (
    OUTPUT_PATH / "role_skill_summary.csv"
)

if not role_skill_file.exists():
    raise FileNotFoundError(role_skill_file)

if not role_summary_file.exists():
    raise FileNotFoundError(role_summary_file)

role_skill_df = pd.read_csv(
    role_skill_file
)

role_skill_summary = pd.read_csv(
    role_summary_file
)

print("✓ Role-skill intelligence loaded.")

print(
    f"✓ Role-skill records: {len(role_skill_df)}"
)

print(
    f"✓ Role summaries: {len(role_skill_summary)}"
)

✓ Role-skill intelligence loaded.
✓ Role-skill records: 21784
✓ Role summaries: 923


In [9]:
# ============================================================
# ORGANIZATION SKILL REQUIREMENT INTELLIGENCE
# ============================================================

organization_skill_demand = (
    role_skill_df
    .groupby("Element Name")
    .agg(
        Role_Count=("O*NET-SOC Code", "nunique"),
        Essential_Count=(
            "Skill_Type",
            lambda x: (x == "Essential").sum()
        ),
        Software_Count=(
            "Skill_Type",
            lambda x: (x == "Software").sum()
        )
    )
    .reset_index()
)

organization_skill_demand = (
    organization_skill_demand
    .sort_values(
        by="Role_Count",
        ascending=False
    )
    .reset_index(drop=True)
)

print("TOP ORGANIZATION-WIDE SKILL REQUIREMENTS")
print("=" * 70)

display(
    organization_skill_demand.head(25)
)

TOP ORGANIZATION-WIDE SKILL REQUIREMENTS


,Element Name,Role_Count,Essential_Count,Software_Count
0,Writing,910,910,0
1,Learning Strategies,910,910,0
2,Monitoring,910,910,0
3,Reading Comprehension,910,910,0
4,Critical Thinking,910,910,0
5,Speaking,910,910,0
6,Mathematics,910,910,0
7,Science,910,910,0
8,Active Learning,910,910,0
9,Active Listening,910,910,0


In [10]:
# ============================================================
# SKILL PRIORITY
# ============================================================

def skill_priority(role_count):

    if role_count >= 100:
        return "HIGH"

    elif role_count >= 50:
        return "MEDIUM"

    else:
        return "LOW"


organization_skill_demand["Priority"] = (
    organization_skill_demand["Role_Count"]
    .apply(skill_priority)
)

print("✓ Skill priority assigned.")

display(
    organization_skill_demand.head(25)
)

✓ Skill priority assigned.


,Element Name,Role_Count,Essential_Count,Software_Count,Priority
0,Writing,910,910,0,HIGH
1,Learning Strategies,910,910,0,HIGH
2,Monitoring,910,910,0,HIGH
3,Reading Comprehension,910,910,0,HIGH
4,Critical Thinking,910,910,0,HIGH
5,Speaking,910,910,0,HIGH
6,Mathematics,910,910,0,HIGH
7,Science,910,910,0,HIGH
8,Active Learning,910,910,0,HIGH
9,Active Listening,910,910,0,HIGH


In [11]:
# ============================================================
# RULE-BASED UPSKILLING RECOMMENDATION ENGINE
# ============================================================

def recommend_from_skill(skill):

    skill_text = str(skill).lower()

    if "python" in skill_text:
        return "Upskill in Python Programming"

    elif "sql" in skill_text:
        return "Upskill in SQL and Database Management"

    elif "machine learning" in skill_text:
        return "Upskill in Machine Learning"

    elif "artificial intelligence" in skill_text:
        return "Upskill in Artificial Intelligence"

    elif "cloud" in skill_text:
        return "Upskill in Cloud Computing"

    elif "docker" in skill_text:
        return "Upskill in Docker and Containerization"

    elif "aws" in skill_text:
        return "Upskill in AWS Cloud"

    elif "data analysis" in skill_text:
        return "Upskill in Data Analysis"

    elif "programming" in skill_text:
        return "Strengthen Programming Skills"

    else:
        return f"Develop {skill} Skills"


organization_skill_demand["Recommendation"] = (
    organization_skill_demand["Element Name"]
    .apply(recommend_from_skill)
)

print("✓ Rule-based recommendations generated.")

display(
    organization_skill_demand[
        [
            "Element Name",
            "Role_Count",
            "Priority",
            "Recommendation"
        ]
    ].head(20)
)

✓ Rule-based recommendations generated.


,Element Name,Role_Count,Priority,Recommendation
0,Writing,910,HIGH,Develop Writing Skills
1,Learning Strategies,910,HIGH,Develop Learning Strategies Skills
2,Monitoring,910,HIGH,Develop Monitoring Skills
3,Reading Comprehension,910,HIGH,Develop Reading Comprehension Skills
4,Critical Thinking,910,HIGH,Develop Critical Thinking Skills
5,Speaking,910,HIGH,Develop Speaking Skills
6,Mathematics,910,HIGH,Develop Mathematics Skills
7,Science,910,HIGH,Develop Science Skills
8,Active Learning,910,HIGH,Develop Active Learning Skills
9,Active Listening,910,HIGH,Develop Active Listening Skills


In [12]:
# ============================================================
# ROLE REQUIREMENT COUNTS
# ============================================================

role_requirement_counts = (
    role_skill_df
    .groupby("Title")["Element Name"]
    .nunique()
    .reset_index()
    .rename(
        columns={
            "Title": "Occupation_Title",
            "Element Name": "Required_Skill_Count"
        }
    )
)

print("✓ Role requirement counts calculated.")

display(
    role_requirement_counts.head(10)
)


✓ Role requirement counts calculated.


,Occupation_Title,Required_Skill_Count
0,Accountants and Auditors,47
1,Actors,22
2,Actuaries,25
3,Acupuncturists,16
4,Acute Care Nurses,19
5,Adapted Physical Education Specialists,18
6,Adhesive Bonding Machine Operators and Tenders,15
7,"Administrative Law Judges, Adjudicators, and H...",23
8,Administrative Services Managers,40
9,"Adult Basic Education, Adult Secondary Educati...",25


In [13]:
# ============================================================
# EMPLOYEE CURRENT-SKILL STATUS
# ============================================================

employee_intelligence["Current_Skills_Status"] = (
    "NOT_AVAILABLE_IN_SOURCE_DATA"
)

employee_intelligence["Skill_Gap_Status"] = (
    "CANNOT_CALCULATE_WITHOUT_CURRENT_SKILLS"
)

employee_intelligence["Skill_Gap"] = (
    "Employee-level current skill data unavailable"
)

print("✓ Employee skill limitation explicitly recorded.")

✓ Employee skill limitation explicitly recorded.


In [14]:
# ============================================================
# EMPLOYEE-LEVEL BUSINESS RECOMMENDATIONS
# ============================================================

def employee_recommendation(row):

    if row["Risk"] == "HIGH":
        return "Retention Review + Targeted Development Plan"

    elif row["Engagement_Level"] == "LOW":
        return "Engagement Improvement + Upskilling Discussion"

    elif row["Risk"] == "MEDIUM":
        return "Career Development and Skill Assessment"

    else:
        return "Continue Development and Monitor Progress"


employee_intelligence["Recommendation"] = (
    employee_intelligence
    .apply(employee_recommendation, axis=1)
)

print("✓ Employee recommendations generated.")

display(
    employee_intelligence[
        [
            "EmployeeNumber",
            "Risk",
            "Engagement_Level",
            "Recommendation"
        ]
    ].head(10)
)

✓ Employee recommendations generated.


,EmployeeNumber,Risk,Engagement_Level,Recommendation
0,1,HIGH,LOW,Retention Review + Targeted Development Plan
1,2,LOW,NaN,Continue Development and Monitor Progress
2,4,MEDIUM,NaN,Career Development and Skill Assessment
3,5,LOW,NaN,Continue Development and Monitor Progress
4,7,LOW,NaN,Continue Development and Monitor Progress
5,8,LOW,NaN,Continue Development and Monitor Progress
6,10,LOW,NaN,Continue Development and Monitor Progress
7,11,LOW,NaN,Continue Development and Monitor Progress
8,12,LOW,NaN,Continue Development and Monitor Progress
9,13,LOW,NaN,Continue Development and Monitor Progress


In [15]:
# ============================================================
# FINAL EMPLOYEE INTELLIGENCE TABLE
# ============================================================

final_employee_intelligence = employee_intelligence[
    [
        "EmployeeNumber",
        "Age",
        "Department",
        "JobRole",
        "MonthlyIncome",
        "Attrition_Prob",
        "Risk",
        "Engagement_Index",
        "Engagement_Level",
        "Current_Skills_Status",
        "Skill_Gap_Status",
        "Skill_Gap",
        "Recommendation"
    ]
].copy()

final_employee_intelligence = (
    final_employee_intelligence
    .rename(
        columns={
            "EmployeeNumber": "Employee_ID",
            "JobRole": "Role",
            "MonthlyIncome": "Monthly_Income",
            "Engagement_Index": "Engagement"
        }
    )
)

print("FINAL EMPLOYEE INTELLIGENCE TABLE")
print("=" * 70)

display(
    final_employee_intelligence.head(15)
)

FINAL EMPLOYEE INTELLIGENCE TABLE


,Employee_ID,Age,Department,Role,Monthly_Income,Attrition_Prob,Risk,Engagement,Engagement_Level,Current_Skills_Status,Skill_Gap_Status,Skill_Gap,Recommendation
0,1,41,Sales,Sales Executive,5993,0.734188,HIGH,33.478613,LOW,NOT_AVAILABLE_IN_SOURCE_DATA,CANNOT_CALCULATE_WITHOUT_CURRENT_SKILLS,Employee-level current skill data unavailable,Retention Review + Targeted Development Plan
1,2,49,Research & Development,Research Scientist,5130,0.010894,LOW,NaN,NaN,NOT_AVAILABLE_IN_SOURCE_DATA,CANNOT_CALCULATE_WITHOUT_CURRENT_SKILLS,Employee-level current skill data unavailable,Continue Development and Monitor Progress
2,4,37,Research & Development,Laboratory Technician,2090,0.571790,MEDIUM,NaN,NaN,NOT_AVAILABLE_IN_SOURCE_DATA,CANNOT_CALCULATE_WITHOUT_CURRENT_SKILLS,Employee-level current skill data unavailable,Career Development and Skill Assessment
3,5,33,Research & Development,Research Scientist,2909,0.121609,LOW,NaN,NaN,NOT_AVAILABLE_IN_SOURCE_DATA,CANNOT_CALCULATE_WITHOUT_CURRENT_SKILLS,Employee-level current skill data unavailable,Continue Development and Monitor Progress
4,7,27,Research & Development,Laboratory Technician,3468,0.375579,LOW,NaN,NaN,NOT_AVAILABLE_IN_SOURCE_DATA,CANNOT_CALCULATE_WITHOUT_CURRENT_SKILLS,Employee-level current skill data unavailable,Continue Development and Monitor Progress
5,8,32,Research & Development,Laboratory Technician,3068,0.090320,LOW,NaN,NaN,NOT_AVAILABLE_IN_SOURCE_DATA,CANNOT_CALCULATE_WITHOUT_CURRENT_SKILLS,Employee-level current skill data unavailable,Continue Development and Monitor Progress
6,10,59,Research & Development,Laboratory Technician,2670,0.093720,LOW,NaN,NaN,NOT_AVAILABLE_IN_SOURCE_DATA,CANNOT_CALCULATE_WITHOUT_CURRENT_SKILLS,Employee-level current skill data unavailable,Continue Development and Monitor Progress
7,11,30,Research & Development,Laboratory Technician,2693,0.145441,LOW,NaN,NaN,NOT_AVAILABLE_IN_SOURCE_DATA,CANNOT_CALCULATE_WITHOUT_CURRENT_SKILLS,Employee-level current skill data unavailable,Continue Development and Monitor Progress
8,12,38,Research & Development,Manufacturing Director,9526,0.061565,LOW,NaN,NaN,NOT_AVAILABLE_IN_SOURCE_DATA,CANNOT_CALCULATE_WITHOUT_CURRENT_SKILLS,Employee-level current skill data unavailable,Continue Development and Monitor Progress
9,13,36,Research & Development,Healthcare Representative,5237,0.038739,LOW,NaN,NaN,NOT_AVAILABLE_IN_SOURCE_DATA,CANNOT_CALCULATE_WITHOUT_CURRENT_SKILLS,Employee-level current skill data unavailable,Continue Development and Monitor Progress


In [16]:
# ============================================================
# SAVE FINAL BUSINESS INTELLIGENCE OUTPUTS
# ============================================================

organization_skill_path = (
    OUTPUT_PATH / "organization_skill_intelligence.csv"
)

recommendation_path = (
    OUTPUT_PATH / "upskilling_recommendations.csv"
)

employee_intelligence_path = (
    PREDICTION_PATH / "employee_intelligence_final.csv"
)

final_employee_intelligence.to_csv(
    employee_intelligence_path,
    index=False
)

organization_skill_demand.to_csv(
    organization_skill_path,
    index=False
)

organization_skill_demand[
    [
        "Element Name",
        "Role_Count",
        "Priority",
        "Recommendation"
    ]
].to_csv(
    recommendation_path,
    index=False
)

print("✓ Final outputs saved.")

print("\nFiles:")
print(f"✓ {employee_intelligence_path.name}")
print(f"✓ {organization_skill_path.name}")
print(f"✓ {recommendation_path.name}")

✓ Final outputs saved.

Files:
✓ employee_intelligence_final.csv
✓ organization_skill_intelligence.csv
✓ upskilling_recommendations.csv


In [17]:
# ============================================================
# NOTEBOOK 11 — FINAL COMPLETION
# ============================================================

print("=" * 70)
print("NOTEBOOK 11 — FINAL BUSINESS INTELLIGENCE COMPLETE")
print("=" * 70)

print("\nEmployee Intelligence:")
print(
    f"✓ {len(final_employee_intelligence)} employees"
)

print(
    f"✓ HIGH risk employees: "
    f"{(final_employee_intelligence['Risk'] == 'HIGH').sum()}"
)

print(
    f"✓ MEDIUM risk employees: "
    f"{(final_employee_intelligence['Risk'] == 'MEDIUM').sum()}"
)

print(
    f"✓ LOW risk employees: "
    f"{(final_employee_intelligence['Risk'] == 'LOW').sum()}"
)

print("\nOrganization Skill Intelligence:")
print(
    f"✓ {len(organization_skill_demand)} skill requirements analyzed"
)

print("\nRecommendations:")
print(
    f"✓ {len(organization_skill_demand)} skill recommendations generated"
)

print("\nFinal Outputs:")
print("✓ employee_intelligence_final.csv")
print("✓ organization_skill_intelligence.csv")
print("✓ upskilling_recommendations.csv")

print("\nData Integrity:")
print("✓ Original employee attrition data used")
print("✓ Original workforce datasets used")
print("✓ No fabricated employee current-skill claims")
print("✓ Employee-level skill gap limitation documented")
print("✓ Department-level engagement used where ID mapping is not validated")

print("\n" + "=" * 70)
print("🎉 ALL 11 NOTEBOOKS COMPLETE")
print("=" * 70)

print("\nNEXT PHASE:")
print("→ FastAPI Backend Integration")
print("→ React Frontend Integration")
print("→ Dashboard + Employee Intelligence UI")

print("=" * 70)

NOTEBOOK 11 — FINAL BUSINESS INTELLIGENCE COMPLETE

Employee Intelligence:
✓ 1470 employees
✓ HIGH risk employees: 68
✓ MEDIUM risk employees: 115
✓ LOW risk employees: 1287

Organization Skill Intelligence:
✓ 144 skill requirements analyzed

Recommendations:
✓ 144 skill recommendations generated

Final Outputs:
✓ employee_intelligence_final.csv
✓ organization_skill_intelligence.csv
✓ upskilling_recommendations.csv

Data Integrity:
✓ Original employee attrition data used
✓ Original workforce datasets used
✓ No fabricated employee current-skill claims
✓ Employee-level skill gap limitation documented
✓ Department-level engagement used where ID mapping is not validated

🎉 ALL 11 NOTEBOOKS COMPLETE

NEXT PHASE:
→ FastAPI Backend Integration
→ React Frontend Integration
→ Dashboard + Employee Intelligence UI
